# Using lifespan models to make predictions on new data ("Transfer")

Created by Saige rutherford adapted by Charlotte Fraza and Johanna Bayer

In this tutorial, we demonstrate how to use large, pretrained normative models to generate predictions for new datasets.  We call this operation a **model transfer**. By transferring, hence applying, the parameters of these models to new data, you can take advantage of the statistical power of large reference cohorts without performing computationally expensive training.

In this script we will demonstrate how to:

1. Download and unzip the pretrained normative models
2. Load and prepare the transfer datasets (in this example case from OpenNeuro)
3. Configure covariates and response variables for the transfer function
4. Adapt the model to a new dataset
5. Run the transfer prediction using the PCNtoolkit

First, we install and import the required libraries. These include some libraries to download the pre-trained models, the pcntookit and some libraries to manipulate data in Python.

In [1]:
# Install the requests library if it's not already installed
try:
    import requests
except ModuleNotFoundError:
    %pip install requests

In [2]:
import os
import requests
import xml.etree.ElementTree as ET
from urllib.parse import unquote
import os
import zipfile
import pcntoolkit as ptk
import pandas as pd
import numpy as np

### 1. Download and unzip the pretrained normative models

In the next step, need to get the models into your computing environment. We are going to download all the models from surfdrive: https://surfdrive.surf.nl/s/Mb6mZyFmJeCaPcZ?dir=/zip and store them in a new folder that we create in your local environment.

We first read out the names of all model folders in the remote directory:

In [4]:

# Define the working directory where all ZIP files will be downloaded
wdir = os.path.abspath("../braincharts/data/")
os.makedirs(wdir, exist_ok=True)

# Define the WebDAV URL and authentication credentials for Surfdrive
BASE_URL = "https://surfdrive.surf.nl/public.php/webdav/zip/"
SHARE_TOKEN = "Mb6mZyFmJeCaPcZ"
PASSWORD = ""

# Send a PROPFIND request to list the contents of the remote directory
headers = {"Depth": "1"}
r = requests.request("PROPFIND", BASE_URL, auth=(SHARE_TOKEN, PASSWORD), headers=headers)

# Parse the XML response returned by the WebDAV server
tree = ET.fromstring(r.content)

# Extract the names of all ZIP files in the remote directory
zip_files = []
for elem in tree.iter():
    if elem.tag.endswith("href"):
        name = unquote(elem.text.split("/")[-1])
        if name.lower().endswith(".zip"):
            zip_files.append(name)


print("Available models:")
print('\n'.join(zip_files))

Available models:
BLRw_ct_DES_lifespan_67K_89sites.zip
BLRw_fa_JHU_lifespan_24K_19sites.zip
BLRw_fc_yeo17_lifespan_21K_40sites.zip
BLRw_sa_DES_lifespan_37K_66sites.zip
BLRw_sa_DK_lifespan_46K_59sites.zip
BLRw_sc_lifespan_67K_89sites.zip
HBR_Sb_ct_DES_lifespan_79K_100sites.zip
HBR_Sb_ct_DK_lifespan_79K_100sites.zip
HBR_Sb_sa_DES_lifespan_37K_66sites.zip
HBR_Sb_sa_DK_lifespan_46K_59sites.zip
HBR_Sb_sc_lifespan_79K_100sites.zip


Then we iterate over the model folder names and download each of them onto your local machine. Execution of this step may take long to run due to the large size (GBs) of the HBR models.

In [ ]:

# Download each ZIP file and save it locally
for fname in zip_files:
    url = BASE_URL + fname
    resp = requests.get(url, auth=(SHARE_TOKEN, PASSWORD), stream=True)

    # Skip files that could not be downloaded successfully
    if resp.status_code != 200:
        continue

    out_path = os.path.join(wdir, fname)
    with open(out_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)


You can now inspect the zipped model files. Your folder should look something like this:

`BLRw_ct_DES_lifespan_67K_89sites.zip` <br>
`HBR_Sb_ct_DES_lifespan_79K_100sites.zip` <br>
`BLRw_fa_JHU_lifespan_24K_19sites.zip` <br>
`HBR_Sb_ct_DK_lifespan_79K_100sites.zip` <br>
`BLRw_sc_lifespan_67K_89sites.zip` <br>
`HBR_Sb_sc_lifespan_79K_100sites.zip` <br>
`BLRw_sa_DES_lifespan_37K_66sites.zip` <br>
`BLRw_sa_DK_lifespan_40K_59sites.zip` <br>
`BLRw_fc_yeo17_lifespan_21K_40sites.zip` <br>
`HBR_Sb_sa_DES_lifespan_37K_66sites.zip` <br>
`HBR_Sb_DK_lifespan_40K_59sites.zip`

Let’s break down what these filenames mean:
- BLRw: Bayesian Linear Regression model (Fraza et al.)
- HBR_Sb: Hierarchical Bayesian Regression model using a SHASH likelihood (de Boer et al., 2024)
- ct: cortical thickness
- fa: fractional anisotropy
- sc: subcortical measures
- DES: Destrieux atlas parcellation
- DK: Desikan–Killiany atlas parcellation
- 67K / 79K / 24K: number of subjects used to train the model
- 89sites / 100sites / 19sites: number of sites in the training dataset

We now define paths to the top (parent) directory and the directory that the models will be extracted into:

In [5]:

# Define the top-level project directory and the data directory (already set as wdir above)
top_dir = os.path.dirname(wdir)  # braincharts/
data_dir = wdir                  # braincharts/data/


Then we extract the models:

In [6]:

# Define paths to the zipped model file and the directory where it will be extracted
zip_path = os.path.join(data_dir, "BLRw_ct_DES_lifespan_67K_89sites.zip")
model_dir = os.path.join(data_dir, "BLRw_ct_DES_lifespan_67K_89sites")

# Unzip the pretrained model if it has not already been extracted
if not os.path.exists(model_dir):
    os.makedirs(model_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(model_dir)


### 2. Load and prepare the transfer datasets (in this example case from OpenNeuro)

This section focuses on preparing the new data for which we want to generate predictions.

In general, the data should be split into two folds:
- The adaptation dataset (_ad_), which is used to estimate site differences between the new data and the pretrained models. This dataset is used only for adaptation and is not included in the final analysis.
- The test dataset (_te_), for which predictions will be generated.

Predictions are made only for the test set.

For this example, we use a publicly available dataset from OpenNeuro. We clone the dataset into a directory called transfer_data. The dataset has already been split into an adaptation set and a test set.

If you plan to use your own data, this is the section you will most likely need to adapt. Your data needs to be split into two folds in a stratified manner, such that the distributions of relevant covariates (e.g., age, sex, site) are represented in both folds. You can use an in-built function for that, or you can create your own custom split. You will also have to make sure that the sex encoding corresponds with the pretrained models (F/M in the BLR case, 0/1 in the HBR SHASH case, with 0 representing female.)

In [7]:

# Clone the adaptation data into the data directory (skip if already present)
transfer_data_dir = os.path.join(data_dir, "transfer_data")
if not os.path.exists(transfer_data_dir):
    ! git clone https://github.com/predictive-clinical-neuroscience/braincharts.git {transfer_data_dir}

# Define paths to the adaptation and test datasets (in this example derived from OpenNeuro), or to your own data.
data_ad_path = os.path.join(transfer_data_dir, "docs", "OpenNeuroTransfer_ct_ad.csv")
data_te_path = os.path.join(transfer_data_dir, "docs", "OpenNeuroTransfer_ct_te.csv")

# Load the adaptation and test datasets into pandas dataframes
df_ad = pd.read_csv(data_ad_path, index_col=0)
df_te = pd.read_csv(data_te_path, index_col=0)


In [8]:
# Create an output directory for transfer predictions
out_dir = model_dir + "_txfr"
os.makedirs(out_dir, exist_ok=True)

### 3. Configure covariates and response variables for the transfer function

In [9]:
# Load the pretrained normative model
model = ptk.NormativeModel.load(model_dir)

# Extract model configuration: batch effects (e.g. site, sex), covariates (e.g. age),
# and response variables (brain measures predicted by the model)
batch_effects = list(model.unique_batch_effects.keys())
covariates = model.covariates
response_vars = model.response_vars

In [10]:
# Combine adaptation and test data into a single dataframe. You don't need to do that necessarily, but we do it here, because it makes making 
# changes to the variables more easy
df = pd.concat([df_ad, df_te], axis=0)

# Recode sex variable to match the format expected by the model (F/M instead of 0/1)
df["sex"] = df["sex"].map({0: "F", 1: "M"})

# Create a subject ID column from the dataframe index
df["sub_id"] = df.index.astype(str)

# Keep only those response variables that are present in the dataframe
# (prevents errors when the model expects variables that are not in the CSV files)
response_vars = [v for v in response_vars if v in df.columns]


### 4 Adapt the model to a new dataset

In [ ]:
# ensure all to use just fitted models
fitted_response_vars = [
    rv for rv in response_vars
    if rv in model.response_vars and model[rv].is_fitted
]

skipped_response_vars = [
    rv for rv in response_vars
    if rv not in fitted_response_vars
]
print(f"Using {len(fitted_response_vars)} fitted response variables")
print("Skipped (not fitted):", skipped_response_vars)

# Create a NormData object to prepare the data for normative modelling
norm_data = ptk.NormData.from_dataframe(
    name="pu25_thickness_txfr",
    dataframe=df,
    batch_effects=batch_effects,
    subject_ids="sub_id",
    response_vars=fitted_response_vars,
    covariates=covariates,
    remove_Nan=True,
    remove_outliers=True,
    z_threshold=10
)

## In case you want to use your own train-test split instead of an automatically generated one, use these lines instead
#train = ptk.NormData.from_dataframe(name="ad", dataframe=df_ad, covariates=covariates, batch_effects=batch_effects, response_vars=response_vars,
# remove_Nan=True,remove_outliers=True,z_threshold=10)
#test = ptk.NormData.from_dataframe(name="te", dataframe=df_te, covariates=covariates, batch_effects=batch_effects, response_vars=response_vars,
# remove_Nan=True,remove_outliers=True,z_threshold=10))
  
# Split the data into adaptation (train) and test sets
train, test = norm_data.train_test_split(splits=[0.5, 0.5])


Using 149 fitted response variables
Skipped (not fitted): ['lh_G&S_frontomargin_thickness']
Process: 36896 - 2026-05-06 08:19:03 - Removed 0 NANs
Process: 36896 - 2026-05-06 08:19:03 - Removed 0 outliers
Process: 36896 - 2026-05-06 08:19:03 - Dataset "pu25_thickness_txfr" created.
    - 546 observations
    - 518 unique subjects
    - 1 covariates
    - 149 response variables
    - 2 batch effects:
    	site (6)
	sex (2)
    


### 5. Run the transfer prediction using the PCNtoolkit

In [ ]:
# Run transfer prediction to adapt the pretrained model and compute deviation scores
transferred_model = model.transfer_predict(train, test, save_dir=out_dir)

Process: 36896 - 2026-05-06 08:19:33 - Transferring models on 149 response variables.
Process: 36896 - 2026-05-06 08:19:33 - Transferring model for rh_S_collat_transv_ant_thickness.
Process: 36896 - 2026-05-06 08:19:33 - Transferring model for lh_S_central_thickness.
Process: 36896 - 2026-05-06 08:19:33 - Transferring model for rh_G_parietal_sup_thickness.
Process: 36896 - 2026-05-06 08:19:33 - Transferring model for lh_S_oc-temp_lat_thickness.
Process: 36896 - 2026-05-06 08:19:33 - Transferring model for rh_G_front_inf-Triangul_thickness.
Process: 36896 - 2026-05-06 08:19:33 - Transferring model for lh_G_occipital_sup_thickness.
Process: 36896 - 2026-05-06 08:19:33 - Transferring model for rh_G_temp_sup-G_T_transv_thickness.
Process: 36896 - 2026-05-06 08:19:33 - Transferring model for lh_S_temporal_inf_thickness.
Process: 36896 - 2026-05-06 08:19:33 - Transferring model for rh_G_temp_sup-Plan_tempo_thickness.
Process: 36896 - 2026-05-06 08:19:33 - Transferring model for lh_G&S_cingul